In [1]:
%load_ext autoreload
%autoreload 2

import os
from glob import glob
from PIL import Image
from IPython.display import display
import sys
import numpy as np
import pandas as pd
import scipy.io as sio
import h5py

from compcats.models import resnet50_vonenet, alexnet_vonenet, cornet_svonenet
from compcats.datasets.image import custom

sys.path.append('../preprint')
from preprint import paths
from preprint import main_analysis as ma

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
PLOTS_DIR = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

MODELS_DIR = '/Users/adipani3/projects/catsel/comparing-category-selectivity/preprint/models'

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


In [2]:
data_dir = '/Users/adipani3/datasets/intracranial/A4'
all_image_files = os.listdir(f'{data_dir}/images')
print(f'Number of image files in {data_dir}/images: {len(all_image_files)}')
image_files_considered = [f for f in all_image_files if f.endswith(('.bmp'))]
image_files_considered.remove('001.bmp') # Remove fixation cross
print(f'Number of image files considered: {len(image_files_considered)}')

# Image labels
image_labels_df = pd.read_csv(f'{data_dir}/image_labels.csv')
# Set column names to 'image_index', 'label', 'category'
image_labels_df.columns = ['label', 'category']
# name index column as 'image_index'
image_labels_df.insert(0, 'image_index', range(0, len(image_labels_df)))
# set image_index as the index of the dataframe
image_labels_df.set_index('image_index', inplace=True)

# Image mapping to the original dataset
image_mapping_df = pd.read_csv(f'{data_dir}/image_mapping.csv')
image_mapping_df = image_mapping_df[
    image_mapping_df['set_number'] == 'set1'
]
# set index column name to 'image_index'
image_mapping_df.insert(0, 'image_index', range(0, len(image_mapping_df)))
image_mapping_df.set_index('image_index', inplace=True)

# Choose only the rows that are in mapping (set 1: Murty185)
image_labels_df = image_labels_df.iloc[:len(image_mapping_df), :]

# Drop first row of both dataframes
image_mapping_df = image_mapping_df.iloc[1:, :]
image_labels_df = image_labels_df.iloc[1:, :]

# Join the two dataframes on the index
image_info_df = image_labels_df.join(image_mapping_df)

assert len(image_info_df) == 185, "Number of images in the dataset should be 185"
image_info_df.head()

Number of image files in /Users/adipani3/datasets/intracranial/A4/images: 872
Number of image files considered: 870


,label,category,set_number,original_image_name,stimulus_image_name
image_index,,,,,
1,foot,body,set1,image1.bmp,002.bmp
2,elbow,body,set1,image10.bmp,003.bmp
3,pool table,object,set1,image100.bmp,004.bmp
4,foot,body,set1,image101.bmp,005.bmp
5,arm,body,set1,image102.bmp,006.bmp


In [3]:
models = [
    resnet50_vonenet.get_model(),
    alexnet_vonenet.get_model(),
    cornet_svonenet.get_model()
]

model_acts = {}
layer = 'vone_block' # V1 block activations
for model in models:
    model_name = model.name
    custom_ds = custom.CustomDataset(
        root_dir=f'{data_dir}/images',
        image_names=image_files_considered,
        dataset_name='intracranial_A4',
        transform=model.transforms
    )
    acts = model.generate_dataset_activations(
        layer,
        custom_ds,
        save_dir=None,
        batch_size=256,
        device='mps',
        num_workers=1
    )
    acts = acts[1:] # Remove first image (fixation cross)
    acts = acts[:185] # Keep only the first 185 images (set 1: Murty185)
    model_acts[model_name] = acts.cpu().numpy()

Neuronal distributions gabor parameters
Model:  VOneResnet50


/Users/adipani3/Softwares/miniconda3/envs/catsel/lib/python3.12/site-packages/vonenet/params.py:59: RuntimeWarning: invalid value encountered in divide
  ny_dist_marg = n_joint_dist / n_joint_dist.sum(axis=1, keepdims=True)


Neuronal distributions gabor parameters
Model:  VOneAlexNet


/Users/adipani3/Softwares/miniconda3/envs/catsel/lib/python3.12/site-packages/vonenet/params.py:59: RuntimeWarning: invalid value encountered in divide
  ny_dist_marg = n_joint_dist / n_joint_dist.sum(axis=1, keepdims=True)


Neuronal distributions gabor parameters
Model:  VOneCORnet-S


/Users/adipani3/Softwares/miniconda3/envs/catsel/lib/python3.12/site-packages/vonenet/params.py:59: RuntimeWarning: invalid value encountered in divide
  ny_dist_marg = n_joint_dist / n_joint_dist.sum(axis=1, keepdims=True)
100%|██████████| 4/4 [00:27<00:00,  6.81s/it]


In [4]:
# Combine DataFrame columns with model responses
for model_name, acts in model_acts.items():
    mat_dict = {
        # Add all DataFrame columns
        'label': image_info_df['label'].values,
        'category': image_info_df['category'].values,
        'set_number': image_info_df['set_number'].values,
        'original_image_name': image_info_df['original_image_name'].values,
        'stimulus_image_name': image_info_df['stimulus_image_name'].values,
        
        # Add model responses (NCHW tensors)
        'vone_block_responses': acts,
    }
    # Save to MAT file
    sio.savemat(f'{model_name}_murty185_responses.mat', mat_dict, do_compression=True)